농장 part

카탈로그용으로 변경 시

1. 셀 5에서 카탈로그용으로 변경

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# 출력 테이블
OUTPUT_TABLE = f"{CATALOG}.gold.farm_status_feature_daily"

In [ ]:
# ─────────────────────────────────────────────────
# 0. 공통 함수: 좌표 기반 farm_id 생성
# ─────────────────────────────────────────────────
def create_farm_id(df):
    """farm_name + farm_address 기반 MD5 → farm_id, 둘 다 NULL이면 farm_id도 NULL"""
    return df.withColumn(
        "farm_id",
        F.when(
            F.col("farm_name").isNull() & F.col("farm_address").isNull(),
            F.lit(None)
        ).otherwise(
            F.md5(F.concat_ws("|",
                F.coalesce(F.col("farm_name"), F.lit("")),
                F.coalesce(F.col("farm_address"), F.lit(""))
            ))
        )
    )

In [ ]:
# ────────────────────────────────────────────────
# 1. CSV 로드
# ─────────────────────────────────────────────────
# 카탈로그용
df = spark.read.table(f"{CATALOG}.silver.farm_master")

df = create_farm_id(df)

In [ ]:
# ─────────────────────────────────────────────────
# 2. poultry_species: 사육종 매핑 및 인코딩
#    매핑 미확정 값은 -1 처리 후 담당자 보고 필요
# ─────────────────────────────────────────────────

# 상위 종 통합 매핑 (확정 전 초안 — 담당자 확인 필요)
SPECIES_MAP = {
    # 닭 계열 → 0
    "육계": 0, "산란계": 0, "토종닭": 0, "씨닭": 0, "닭": 0,
    # 오리 계열 → 1
    "육용오리": 1, "씨오리": 1, "토종오리": 1, "오리": 1,
    # 기타 → 2
    "거위": 2, "칠면조": 2, "메추리": 2,
    "관상조류": 2, "타조": 2, "기러기": 2, "꿩": 2,
}

# 복수 종(혼합) 처리 기준 미확정 → 담당자 결정 후 반영
# 현재는 -1로 처리하고 로그에 기록

species_map_expr = F.create_map(
    *[x for pair in [(F.lit(k), F.lit(v)) for k, v in SPECIES_MAP.items()] for x in pair]
)

df = (
    df
    .withColumnRenamed("livestock_name", "poultry_species_raw")
    .withColumn(
        "poultry_species",
        F.coalesce(
            species_map_expr[F.col("poultry_species_raw")],
            F.lit(-1)  # NULL 또는 매핑 없는 값 → -1
        )
    )
)

# [담당자 보고] 매핑 안 된 고유값 출력
unmapped = (
    df
    .filter(F.col("poultry_species") == -1)
    .groupBy("poultry_species_raw")
    .count()
    .orderBy(F.desc("count"))
)
print("=== [담당자 확인] 매핑 미확정 사육종 목록 ===")
unmapped.show(truncate=False)

In [ ]:
# ─────────────────────────────────────────────────
# 3. farm_id별 복수 종 확인
#    처리 기준 미확정 → 담당자 확인 필요
# ─────────────────────────────────────────────────

# 동일 farm_id에 서로 다른 사육종이 있는 경우 집계
multi_species = (
    df
    .groupBy("farm_id")
    .agg(F.countDistinct("poultry_species_raw").alias("species_cnt"))
    .filter(F.col("species_cnt") > 1)
)
multi_count = multi_species.count()
if multi_count > 0:
    print(f"=== [담당자 확인] 복수 사육종 farm_id: {multi_count}건 — 처리 기준 확정 필요 ===")
    multi_species.show(truncate=False)

In [ ]:
# ─────────────────────────────────────────────────
# 4. flock_size: 사육두수 정제
# ─────────────────────────────────────────────────
df = (
    df
    .withColumn(
        "flock_size",
        # 쉼표·공백·단위 문자 제거 후 정수 변환; 변환 불가 → NULL
        F.regexp_replace(F.col("head_count").cast("string"), r"[,\s수마리]", "")
         .cast("long")
    )
)

# [담당자 보고] 음수, 0, NULL
invalid_flock = (
    df
    .filter(F.col("flock_size").isNull() | (F.col("flock_size") <= 0))
    .groupBy(
        F.when(F.col("flock_size").isNull(), "NULL")
         .when(F.col("flock_size") == 0, "0")
         .otherwise("음수").alias("상태")
    )
    .count()
)
print("=== [담당자 확인] flock_size 이상값 ===")
invalid_flock.show()

# 상위 0.1% 이상치 확인 (제거 여부는 담당자 결정)
threshold = df.approxQuantile("flock_size", [0.999], 0.001)[0]
outlier_count = df.filter(F.col("flock_size") > threshold).count()
print(f"=== [담당자 확인] flock_size 상위 0.1% 기준값: {threshold:,.0f}, 해당 행 수: {outlier_count} ===")

In [ ]:
# ─────────────────────────────────────────────────
# 5. farm_count_3km: 반경 3km 내 농장 수
#    좌표계 WGS84(EPSG:4326) 가정 — 실제 좌표계 확인 필요
# ─────────────────────────────────────────────────

# 좌표가 유효한 농장만 거리 계산에 사용
valid_coord = df.filter(
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull()
).select("farm_id", "latitude", "longitude").dropDuplicates(["farm_id"])

# Haversine 거리 계산 (km)
# 자기 자신 제외: farm_id가 다른 농장만 카운트
base = valid_coord.alias("base")
comp = valid_coord.alias("comp")

nearby = (
    base.crossJoin(comp)
    .filter(F.col("base.farm_id") != F.col("comp.farm_id"))
    .withColumn(
        "dist_km",
        F.lit(6371) * F.acos(
            F.sin(F.radians("base.latitude")) * F.sin(F.radians("comp.latitude")) +
            F.cos(F.radians("base.latitude")) * F.cos(F.radians("comp.latitude")) *
            F.cos(F.radians("comp.longitude") - F.radians("base.longitude"))
        )
    )
    .filter(F.col("dist_km") <= 3.0)
    .groupBy(F.col("base.farm_id").alias("farm_id"))
    .agg(F.countDistinct("comp.farm_id").alias("farm_count_3km"))
)

# 전체 농장 기준 LEFT JOIN — 반경 내 농장 없으면 0, 좌표 NULL이면 NULL 유지
df = (
    df
    .join(nearby, "farm_id", "left")
    .withColumn(
        "farm_count_3km",
        F.when(
            F.col("latitude").isNull() | F.col("longitude").isNull(),
            F.lit(None).cast("long")
        ).otherwise(F.coalesce(F.col("farm_count_3km"), F.lit(0)))
    )
)

In [ ]:
# ─────────────────────────────────────────────────
# 6. reference_date 타입 보정 및 최종 컬럼 선택
# ─────────────────────────────────────────────────
df = df.withColumn("reference_date", F.col("reference_date").cast("date"))

farm_status_feature = df.select(
    "farm_id",
    "county",
    "farm_name",
    "farm_address",
    "latitude",
    "longitude",
    "poultry_species_raw",   # 원본 보존
    "poultry_species",       # 인코딩 값 (0/1/2/-1)
    "flock_size",
    "farm_count_3km",
    "reference_date",        # base_date → reference_date
    "outbreak_date",
    "label_infected",
)

In [ ]:
# ─────────────────────────────────────────────────
# 6.5 중복 제거 + flock_size farm_id 평균으로 덮어씌우기
# 중복 기준: farm_id + reference_date
# flock_size NULL → 0으로 치환 후 평균 계산
# ─────────────────────────────────────────────────

# farm_id별 flock_size 평균 (NULL → 0 치환)
flock_avg = (
    df
    .withColumn("flock_size_filled", F.coalesce(F.col("flock_size"), F.lit(0)))
    .groupBy("farm_id")
    .agg(F.avg("flock_size_filled").cast("long").alias("flock_size_avg"))
)

# flock_size를 평균값으로 덮어씌우기
df = (
    df
    .join(flock_avg, "farm_id", "left")
    .drop("flock_size")
    .withColumnRenamed("flock_size_avg", "flock_size")
)

# farm_id + reference_date 기준 중복 제거
df = df.dropDuplicates(["farm_id", "reference_date"])

In [ ]:
# ─────────────────────────────────────────────────
# 7. gold 테이블 저장
# ─────────────────────────────────────────────────
(
    farm_status_feature
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE)
)

print(OUTPUT_TABLE)